In [39]:
import pandas as pd
import numpy as np
import torch
import gc
import os
from datasets import Dataset, DatasetDict, load_dataset, concatenate_datasets
from transformers import (
    BertConfig, 
    BertTokenizerFast, 
    BertForQuestionAnswering,
    BertForMaskedLM,
    DataCollatorForLanguageModeling, 
    TrainingArguments, 
    Trainer,
    DefaultDataCollator
)
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm

In [40]:
OUTPUT_DIR = "custom_bert_scratch"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Using Device: {DEVICE}")
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

🚀 Using Device: cuda


In [41]:
print("\n" + "="*40)
print("🏗️ Building Custom MEDIUM BERT (DistilBERT-like)")
print("="*40)

# Define a MEDIUM Model (Close to DistilBERT)
# This is the "Sweet Spot" for training from scratch on limited compute.
config = BertConfig(
    vocab_size=30522,
    hidden_size=512,       # Increased Width (256 -> 512)
    num_hidden_layers=6,   # Keep Depth at 6 (Efficient)
    num_attention_heads=8, # Increased Heads (4 -> 8)
    intermediate_size=2048,
    max_position_embeddings=512,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1
)

# ---------------------------------------------------------
# STEP 1: Masked Language Modeling (Scaled Up)
# ---------------------------------------------------------
print("\n🔄 Step 1: Pre-training on Wikitext-103 (Larger Subset)...")

lines = []

# 1. Load Wikitext-103
print("   Downloading/Loading Wikitext-103 (Subset)...")
try:
    # 20% of Wikitext-103
    wiki_dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:20%]")
    print(f"   ✅ Loaded Wikipedia Data: {len(wiki_dataset)} lines")
except Exception as e:
    print(f"   ❌ Failed to load Wikitext-103: {e}")
    wiki_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# 2. Load Domain Data
if os.path.exists("dapt_corpus.txt"):
    with open("dapt_corpus.txt", "r", encoding="utf-8") as f:
        lines.extend([line.strip() for line in f.readlines() if len(line.strip()) > 0])

# 3. Load Training Data Text
try:
    df_train = pd.read_csv("training_data_en.csv", sep=";", engine="python")
    lines.extend(df_train["Text"].dropna().tolist())
except: pass

domain_dataset = Dataset.from_dict({"text": lines})
print(f"   ✅ Loaded Domain+Train Data: {len(domain_dataset)} lines")

# Combine
full_mlm_dataset = concatenate_datasets([wiki_dataset, domain_dataset])
full_mlm_dataset = full_mlm_dataset.filter(lambda x: len(x["text"]) > 20)

print(f"   🚀 Total Training Sequences: {len(full_mlm_dataset)}")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_mlm = full_mlm_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)

# Initialize MLM Model
mlm_model = BertForMaskedLM(config).to(DEVICE)
print(f"   Model Size: {mlm_model.num_parameters()/1e6:.1f}M Parameters")

mlm_args = TrainingArguments(
    output_dir="custom_bert_mlm_checkpoints",
    overwrite_output_dir=True,
    num_train_epochs=8,         # 8 Epochs for larger model
    per_device_train_batch_size=32,
    save_strategy="no",
    learning_rate=1e-4,         # Slightly lower LR for larger model stability
    weight_decay=0.01,
    fp16=True,
    report_to="none",
    disable_tqdm=False
)

mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_args,
    train_dataset=tokenized_mlm,
    data_collator=data_collator,
)

mlm_trainer.train()
print("✅ MLM Pre-training Complete.")

# ---------------------------------------------------------
# STEP 2: Initialize QA Model
# ---------------------------------------------------------
print("\n🔄 Step 2: Transferring Encoder to QA Model...")

model = BertForQuestionAnswering(config).to(DEVICE)
model.bert.load_state_dict(mlm_model.bert.state_dict())

print("✅ Model Initialized.")

del mlm_model, mlm_trainer, full_mlm_dataset, tokenized_mlm
gc.collect()
torch.cuda.empty_cache()


🏗️ Building Custom MEDIUM BERT (DistilBERT-like)

🔄 Step 1: Pre-training on Wikitext-103 (Larger Subset)...
   Downloading/Loading Wikitext-103 (Subset)...
   ✅ Loaded Wikipedia Data: 360270 lines
   ✅ Loaded Domain+Train Data: 4303 lines
   🚀 Total Training Sequences: 220595
   Model Size: 35.1M Parameters


Step,Training Loss
500,7.263600
1000,6.831200
1500,6.692700
2000,6.610600
2500,6.512000
3000,6.459300
3500,6.371200
4000,6.282900
4500,6.078600
5000,5.813000


✅ MLM Pre-training Complete.

🔄 Step 2: Transferring Encoder to QA Model...
✅ Model Initialized.


In [42]:
TRAIN_FILE = "training_data_en.csv"

def load_data(file):
    df = pd.read_csv(file, sep=";", engine="python")
    data = []
    for _, row in df.dropna(subset=["Text", "Question"]).iterrows():
        if "Answer" in row and isinstance(row["Answer"], str):
            ans = row["Answer"].strip()
            # Strict Filter (Sarang Method) to give it a fighting chance
            if row["Text"].find(ans) != -1:
                data.append({
                    "id": str(row["ID"]), "context": row["Text"], "question": row["Question"],
                    "answers": {"text": [ans], "answer_start": [row["Text"].find(ans)]}
                })
    return Dataset.from_list(data)

full_data = load_data(TRAIN_FILE)
split = full_data.train_test_split(test_size=0.2, seed=42)
qa_dataset = DatasetDict({"train": split["train"], "validation": split["test"]})

In [43]:
def preprocess_function(examples):
    inputs = tokenizer(
        examples["question"], examples["context"],
        max_length=384, truncation="only_second", stride=128,
        return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length",
    )
    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answers = examples["answers"][sample_idx]
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])
        sequence_ids = inputs.sequence_ids(i)
        
        idx = 0
        while sequence_ids[idx] != 1: idx += 1
        context_start = idx
        while sequence_ids[idx] == 1: idx += 1
        context_end = idx - 1

        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0); end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char: idx += 1
            start_positions.append(idx - 1)
            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char: idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

tokenized_datasets = qa_dataset.map(preprocess_function, batched=True, remove_columns=qa_dataset["train"].column_names)

Map: 100%|██████████| 398/398 [00:00<00:00, 4023.68 examples/s]


In [44]:
print("\n" + "="*40)
print("🚀 Training QA (Fine-Tuning)...")
print("="*40)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=30,      # Reduced from 100 to 30 to prevent Overfitting
    learning_rate=5e-5,
    per_device_train_batch_size=16, # Increased Batch Size for stability
    weight_decay=0.05,        # Increased Weight Decay for Regularization
    fp16=True,
    save_strategy="no",
    report_to="none",
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=DefaultDataCollator(),
)

trainer.train()
print("✅ Training Complete.")

C:\Users\kisho\AppData\Local\Temp\ipykernel_24020\3509385044.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



🚀 Training QA (Fine-Tuning)...


Step,Training Loss
100,2.057900
200,1.222200
300,0.877400
400,0.633900
500,0.437200
600,0.300400
700,0.217000
800,0.169100
900,0.127900
1000,0.107600


✅ Training Complete.


In [45]:
print("\n" + "="*40)
print("📊 Evaluating Custom Model Results")
print("="*40)

def predict(context, question):
    inputs = tokenizer(question, context, return_tensors="pt", max_length=384, truncation="only_second").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    
    start_logits = outputs.start_logits
    end_logits = outputs.end_logits
    
    s_idx = torch.argmax(start_logits)
    e_idx = torch.argmax(end_logits)
    
    if e_idx < s_idx: return ""
    
    tokens = inputs.input_ids[0][s_idx : e_idx + 1]
    return tokenizer.decode(tokens, skip_special_tokens=True)

# Run Inference on Validation Set
preds = []
truths = []
print("Generating predictions...")

for row in tqdm(qa_dataset["validation"]):
    pred = predict(row["context"], row["question"])
    preds.append(pred)
    truths.append(row["answers"]["text"][0])

# Metrics
sas_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
em = sum([1 if p.strip() == t.strip() else 0 for p, t in zip(preds, truths)]) / len(truths)

with torch.no_grad():
    t_emb = sas_model.encode(truths, convert_to_tensor=True)
    p_emb = sas_model.encode(preds, convert_to_tensor=True)
    sas = torch.diag(util.cos_sim(t_emb, p_emb)).mean().item()

print(f"\n📉 CUSTOM (FROM SCRATCH) RESULTS:")
print(f"Exact Match (EM): {em*100:.2f}%")
print(f"Semantic Score (SAS): {sas:.4f}")
print("\n(Note: Low scores are EXPECTED. This proves Transfer Learning is necessary.)")


📊 Evaluating Custom Model Results
Generating predictions...


100%|██████████| 398/398 [00:06<00:00, 64.08it/s]



📉 CUSTOM (FROM SCRATCH) RESULTS:
Exact Match (EM): 9.55%
Semantic Score (SAS): 0.6586

(Note: Low scores are EXPECTED. This proves Transfer Learning is necessary.)
